# Robust BO with Input Perturbations

このNotebookのrobustnessはtraining observationのoutlier robustnessではなく、**実行時にnominal designから入力がずれるdecision robustness**です。

```text
nominal candidate X
  → perturbation scenarios X + δ_w
  → surrogate posterior
  → posterior samples
  → scenario-risk aggregation
  → MC acquisition
  → optimize nominal X
```

scenario axisはcandidate `q`、MC sample axis、optimizer restart axisとは別です。

## 1. Imports and reproducibility

In [ ]:
import torch
from botorch.acquisition.logei import qLogNoisyExpectedImprovement
from botorch.acquisition.risk_measures import Expectation as BoTorchExpectation
from botorch.fit import fit_gpytorch_mll
from botorch.models.transforms.input import InputPerturbation
from botorch.sampling.normal import SobolQMCNormalSampler

from robotorchan.models import SingleTaskGP
from robotorchan.objectives import CVaR, Expectation, MeanVariance, WorstCase
from robotorchan.optim import optimize_acqf

torch.set_default_dtype(torch.double)
torch.manual_seed(0)

## 2. Nominal design problem and execution perturbations

候補は1次元nominal designです。実行時には`-0.04, 0, +0.04`のずれが起きるとします。

In [ ]:
def response(X: torch.Tensor) -> torch.Tensor:
    return 1.0 - 6.0 * (X - 0.62).square() + 0.12 * torch.sin(8 * torch.pi * X)


perturbations = torch.tensor([[-0.04], [0.0], [0.04]])
n_w = perturbations.shape[0]
bounds = torch.tensor([[0.08], [0.92]])
train_X = torch.linspace(0.10, 0.90, 11).unsqueeze(-1)
train_Y = response(train_X)

print("nominal input dimension:", train_X.shape[-1])
print("number of perturbation scenarios:", n_w)

## 3. InputPerturbation expands candidates before posterior evaluation

physical perturbationはraw/public design spaceで適用します。`n_w`を入力次元へ追加するのではありません。

In [ ]:
input_transform = InputPerturbation(perturbation_set=perturbations)
model = SingleTaskGP(
    train_X,
    train_Y,
    input_transform=input_transform,
)
fit_gpytorch_mll(model.make_mll())
model.eval()

nominal_X = torch.tensor([[0.55]])
expanded_X = input_transform(nominal_X)
print("nominal shape:", nominal_X.shape)
print("expanded scenario shape:", expanded_X.shape)
print("expanded scenarios:\n", expanded_X)

## 4. Posterior sampling keeps MC and scenario axes distinct

`InputPerturbation`によりposteriorのq-like row axisへscenarioが展開されます。risk objectiveが`n_w`個を1つのnominal candidate utilityへ戻します。

In [ ]:
sampler = SobolQMCNormalSampler(torch.Size([64]), seed=101)
posterior = model.posterior(nominal_X)
samples = sampler(posterior)

print("posterior mean shape:", posterior.mean.shape)
print("MC sample shape:", samples.shape)
assert samples.shape[0] == 64
assert samples.shape[-2] == n_w

## 5. Scenario-risk semantics

robotorchanのrisk measuresはscenario valuesの最終軸を集約する汎用APIです。BoTorchの`MCAcquisitionObjective`とは別の責務です。

In [ ]:
scenario_values = response(expanded_X).squeeze(-1)
risk_values = {
    "expectation": Expectation()(scenario_values),
    "mean_variance": MeanVariance(risk_weight=2.0)(scenario_values),
    "worst_case": WorstCase()(scenario_values),
    "cvar": CVaR(alpha=0.8)(scenario_values),
}

print("scenario values:", scenario_values)
for name, value in risk_values.items():
    print(name, value.item())

## 6. Robust acquisition uses a scenario-aware BoTorch objective

E2E candidate optimizationではBoTorch-native `Expectation(n_w=...)`をMC acquisitionへ渡します。これによりscenario axisを平均してnominal candidate utilityに戻します。

In [ ]:
robust_objective = BoTorchExpectation(n_w=n_w)
acquisition = qLogNoisyExpectedImprovement(
    model=model,
    X_baseline=train_X,
    sampler=SobolQMCNormalSampler(torch.Size([128]), seed=202),
    objective=robust_objective,
    prune_baseline=False,
)
candidate, value = optimize_acqf(
    acq_function=acquisition,
    bounds=bounds,
    q=1,
    optimizer="botorch",
    num_restarts=4,
    raw_samples=64,
)

assert candidate.shape == torch.Size([1, 1])
print("robust nominal candidate:", candidate)
print("acquisition value:", value)

## 7. Candidate evaluation remains scenario-aware

optimizerが返すのは`n_w`行ではなくnominal candidate 1点です。実行時riskを確認するときだけperturbation scenariosへ展開します。

In [ ]:
candidate_scenarios = input_transform(candidate)
candidate_values = response(candidate_scenarios).squeeze(-1)
expected_value = Expectation()(candidate_values)
worst_value = WorstCase()(candidate_values)

assert candidate_scenarios.shape == torch.Size([n_w, 1])
print("nominal candidate:", candidate.squeeze(-1))
print("scenario responses:", candidate_values)
print("scenario expectation:", expected_value.item())
print("scenario worst case:", worst_value.item())

## 8. Update data with what was actually observed

robust acquisitionのrisk utilityをtraining targetへ自動的に書き換えません。実験・シミュレーションで何を観測するかはdata-generating processの契約です。ここではnominal responseを1観測として追加します。

In [ ]:
observed_Y = response(candidate)
updated_X = torch.cat([train_X, candidate], dim=0)
updated_Y = torch.cat([train_Y, observed_Y], dim=0)

assert updated_X.shape[0] == train_X.shape[0] + 1
print("observed nominal response:", observed_Y.item())

## 9. Initialization remains nominal

optimizer initial conditionsは`[num_restarts, q, d]`です。scenario数`n_w`を`q`や`d`へ追加しません。

```text
IC: [restart, q, d]
          ↓
InputPerturbation
          ↓
posterior scenario rows: q * n_w
```

candidate constraintsも原則としてnominal design coordinatesに対して定義します。

## 10. Protected structural dimensions

Mixed / MultiTask / MultiFidelityなどでは、categorical、task、fidelity、context、hierarchy等のstructural coordinatesを物理perturbationの対象にしません。continuous design/environmental coordinatesだけを揺らし、protected dimensionsのunionを保持します。

## 11. Robustness layers are independent

| Problem | Layer |
| --- | --- |
| training outlier / heavy tail | robust observation surrogate |
| uncertain measured X | uncertain-input surrogate |
| execution-time X deviation | input perturbation scenarios |
| expectation / CVaR / worst case | decision risk aggregation |
| noisy Y | observation-noise / noisy acquisition semantics |

robust surrogateを使うだけでCVaR optimizationにはならず、risk objectiveを使うだけでtraining outlierへ頑健になるわけでもありません。

## 12. TuRBO and high-dimensional boundaries

TuRBOのtrust regionはnominal public input spaceにあります。scenario axisはTuRBO dimensionや`batch_size`へ追加しません。PCA/PLS等でも物理perturbationはraw design spaceへ適用してからmodel-owned reductionへ進めます。latent offsetを物理的不確かさとして暗黙に扱いません。

## 13. Related documentation

- `docs/development/input_perturbation_contract.md`
- `docs/optimization/objective-posterior-transform.md`
- `docs/theory/14_robust_gaussian_process.md`
- `docs/theory/16_uncertain_input_gp.md`
- `29_objective_posterior_transform.ipynb`
- `34_turbo_trust_region.ipynb`

Robust BOでは「何が不確かか」と「どの軸をどこで集約するか」を明示することが最重要です。